In [13]:
import pandas as pd
import numpy as np
import joblib
import os
import glob
from pathlib import Path

In [14]:
def load_sample_data(sample_file="samples.txt"):
    """Load the sample data for inference"""
    try:
        # Try different separators
        for sep in ['\t', ',', ' ']:
            try:
                df = pd.read_csv(sample_file, sep=sep)
                if df.shape[1] > 1:  # Found the right separator
                    break
            except:
                continue
        
        print(f"✅ Loaded sample data: {df.shape}")
        print(f"Columns: {list(df.columns[:5])}..." if len(df.columns) > 5 else f"Columns: {list(df.columns)}")
        return df
    except Exception as e:
        print(f"❌ Error loading sample data: {e}")
        return None

In [15]:
def find_models():
    """Find all trained model files in current directory"""
    model_files = []
    
    # Look for .pkl and .joblib files
    for pattern in ["*.pkl", "*.joblib"]:
        model_files.extend(glob.glob(pattern))
    
    if not model_files:
        print("❌ No model files found in current directory!")
        return []
    
    print(f"📦 Found {len(model_files)} model files:")
    for i, model_file in enumerate(model_files, 1):
        print(f"  {i}. {model_file}")
    
    return model_files


In [16]:
def prepare_features(df):
    """Prepare features for inference and detect target columns"""
    # Common target column names
    target_columns = ['avg7_calingiri', 'avg7_lancer', 'avg_halberd']
    
    # Check if any target column exists
    found_target = None
    actual_values = None
    
    for target_col in target_columns:
        if target_col in df.columns:
            found_target = target_col
            actual_values = df[target_col].values
            print(f"✅ Found target column: {target_col}")
            break
    
    if found_target is None:
        print("ℹ️  No target column found - running inference only")
    
    # Remove ID and target columns for features
    columns_to_remove = ['ID', 'id', 'Id', 'sample_id', 'Sample_ID'] + target_columns
    features_df = df.copy()
    
    for col in columns_to_remove:
        if col in features_df.columns:
            if col not in target_columns:  # Only print for ID columns
                print(f"Removing ID column: {col}")
            features_df = features_df.drop(columns=[col])
    
    print(f"Features shape: {features_df.shape}")
    return features_df, found_target, actual_values

In [17]:
def calculate_metrics(actual, predicted, model_name):
    """Calculate performance metrics"""
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    
    mse = mean_squared_error(actual, predicted)
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)
    rmse = np.sqrt(mse)
    
    return {
        'Model': model_name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }


In [18]:
def create_comparison_plots(results_df, actual_values, target_name):
    """Create actual vs predicted plots"""
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # Get prediction columns (exclude Sample_ID and actual values)
    pred_columns = [col for col in results_df.columns if col not in ['Sample_ID', f'Actual_{target_name}']]
    
    # Create subplots
    n_models = len(pred_columns)
    n_cols = min(3, n_models)
    n_rows = (n_models + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    if n_models == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx, model_col in enumerate(pred_columns):
        row = idx // n_cols
        col = idx % n_cols
        ax = axes[row][col] if n_rows > 1 else axes[col]
        
        # Scatter plot
        ax.scatter(actual_values, results_df[model_col], alpha=0.6, s=50)
        
        # Perfect prediction line
        min_val = min(min(actual_values), min(results_df[model_col]))
        max_val = max(max(actual_values), max(results_df[model_col]))
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, alpha=0.8)
        
        ax.set_xlabel('Actual Values')
        ax.set_ylabel('Predicted Values')
        ax.set_title(f'{model_col}')
        ax.grid(True, alpha=0.3)
    
    # Hide empty subplots
    for idx in range(n_models, n_rows * n_cols):
        row = idx // n_cols
        col = idx % n_cols
        if n_rows > 1:
            axes[row][col].set_visible(False)
        else:
            axes[col].set_visible(False)
    
    plt.suptitle(f'Actual vs Predicted - {target_name}', fontsize=16)
    plt.tight_layout()
    plt.savefig(f'actual_vs_predicted_{target_name}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"📊 Comparison plots saved as: actual_vs_predicted_{target_name}.png")
    """Main inference function"""
    print("🔮 Model Inference Script")
    print("=" * 40)
    
    # Load sample data
    sample_df = load_sample_data()
    if sample_df is None:
        return
    

In [19]:

def run_inference():
    """Main inference function"""
    print("🔮 Model Inference Script")
    print("=" * 40)
    
    # Load sample data
    sample_df = load_sample_data()
    if sample_df is None:
        return
    
    # Prepare features and check for target values
    X_sample, target_name, actual_values = prepare_features(sample_df.copy())
    
    # Find models
    model_files = find_models()
    if not model_files:
        return
    
    # Results storage
    all_predictions = {}
    
    print(f"\n🚀 Running inference on {len(model_files)} models...")
    print("-" * 40)
    
    # Run inference on each model
    for model_file in model_files:
        try:
            # Load model
            model = joblib.load(model_file)
            
            # Handle different model storage formats
            if isinstance(model, dict):
                if 'model' in model:
                    actual_model = model['model']
                    selected_features = model.get('selected_features', None)
                    
                    # Use selected features if available (for genetic algorithm models)
                    if selected_features is not None:
                        X_inference = X_sample.iloc[:, selected_features]
                        print(f"📊 {model_file}: Using {len(selected_features)} selected features")
                    else:
                        X_inference = X_sample
                else:
                    actual_model = model
                    X_inference = X_sample
            else:
                actual_model = model
                X_inference = X_sample
            
            # Make predictions
            predictions = actual_model.predict(X_inference)
            
            # Store predictions
            model_name = Path(model_file).stem
            all_predictions[model_name] = predictions
            
            print(f"✅ {model_file}: {len(predictions)} predictions made")
            print(f"   Sample predictions: {predictions[:3].round(4)}")
            
        except Exception as e:
            print(f"❌ Error with {model_file}: {e}")
            continue
    
    if not all_predictions:
        print("❌ No successful predictions made!")
        return
    
    # Create results dataframe
    print(f"\n📊 Creating results summary...")
    
    # Add sample IDs if they exist
    if 'ID' in sample_df.columns:
        results_df = pd.DataFrame({'Sample_ID': sample_df['ID']})
    else:
        results_df = pd.DataFrame({'Sample_ID': range(1, len(sample_df) + 1)})
    
    # Add actual values if available
    if actual_values is not None:
        results_df[f'Actual_{target_name}'] = actual_values
    
    # Add predictions from each model
    for model_name, predictions in all_predictions.items():
        results_df[model_name] = predictions
    
    # Calculate ensemble prediction (mean of all models)
    prediction_columns = [col for col in results_df.columns if col not in ['Sample_ID', f'Actual_{target_name}']]
    results_df['Ensemble_Mean'] = results_df[prediction_columns].mean(axis=1)
    results_df['Ensemble_Median'] = results_df[prediction_columns].median(axis=1)
    
    # Display results
    print(f"\n📋 Prediction Results:")
    print("-" * 40)
    print(results_df.head(10).to_string(index=False))
    
    if len(results_df) > 10:
        print(f"\n... and {len(results_df) - 10} more samples")
    
    # Save results
    output_file = "inference_results.csv"
    results_df.to_csv(output_file, index=False)
    print(f"\n💾 Results saved to: {output_file}")
    
    # If actual values are available, calculate metrics
    if actual_values is not None:
        print(f"\n🎯 Performance Metrics ({target_name}):")
        print("=" * 50)
        
        metrics_list = []
        eval_columns = prediction_columns + ['Ensemble_Mean', 'Ensemble_Median']
        
        for col in eval_columns:
            metrics = calculate_metrics(actual_values, results_df[col], col)
            metrics_list.append(metrics)
            
            print(f"{col}:")
            print(f"  R² Score: {metrics['R2']:.4f}")
            print(f"  MSE:      {metrics['MSE']:.4f}")
            print(f"  RMSE:     {metrics['RMSE']:.4f}")
            print(f"  MAE:      {metrics['MAE']:.4f}")
            print()
        
        # Save metrics
        metrics_df = pd.DataFrame(metrics_list)
        metrics_df = metrics_df.sort_values('R2', ascending=False)
        metrics_df.to_csv("inference_metrics.csv", index=False)
        print(f"📊 Performance metrics saved to: inference_metrics.csv")
        
        # Show best performing model
        best_model = metrics_df.iloc[0]
        print(f"🏆 Best Performing Model: {best_model['Model']} (R² = {best_model['R2']:.4f})")
        
        # Create comparison plots
        try:
            create_comparison_plots(results_df, actual_values, target_name)
        except Exception as e:
            print(f"⚠️  Could not create plots: {e}")
    
    # Summary statistics
    print(f"\n📈 Prediction Summary:")
    print("-" * 40)
    all_pred_cols = prediction_columns + ['Ensemble_Mean']
    if actual_values is not None:
        all_pred_cols = [f'Actual_{target_name}'] + all_pred_cols
    
    for col in all_pred_cols:
        predictions = results_df[col]
        print(f"{col}:")
        print(f"  Mean: {predictions.mean():.4f}")
        print(f"  Std:  {predictions.std():.4f}")
        print(f"  Min:  {predictions.min():.4f}")
        print(f"  Max:  {predictions.max():.4f}")
        print()
    
    return results_df



In [20]:
if __name__ == "__main__":
    results = run_inference()

🔮 Model Inference Script
✅ Loaded sample data: (2, 33049)
Columns: ['ID', 'SNOO_500610_1', 'SNOO_500610_2', 'SNOO_505150_1', 'SNOO_104700A_1']...
ℹ️  No target column found - running inference only
Removing ID column: ID
Features shape: (2, 33048)
📦 Found 6 model files:
  1. Calingiri_Decision_Tree_K7.pkl
  2. Halberd_Decision_Tree_K7.pkl
  3. Lancer_Decision_Tree_K7.pkl
  4. Calingiri_GradientBoosting_k3_best_model.joblib
  5. Halberd_GradientBoosting_k3_best_model.joblib
  6. Lancer_GradientBoosting_k3_best_model.joblib

🚀 Running inference on 6 models...
----------------------------------------
✅ Calingiri_Decision_Tree_K7.pkl: 2 predictions made
   Sample predictions: [4.5638 2.335 ]
✅ Halberd_Decision_Tree_K7.pkl: 2 predictions made
   Sample predictions: [5.     4.7553]
✅ Lancer_Decision_Tree_K7.pkl: 2 predictions made
   Sample predictions: [4.7423 3.1429]
📊 Calingiri_GradientBoosting_k3_best_model.joblib: Using 16473 selected features
✅ Calingiri_GradientBoosting_k3_best_model.